# pyannote/embedding with OpenVINO — CPU/GPU Enablement

This notebook walks through enabling `pyannote/embedding` speaker-embedding inference with OpenVINO, step by step:

1. Create the `ov_pyan` Conda environment
2. Authenticate with Hugging Face
3. Convert the PyTorch model to OpenVINO IR
4. Run sample embedding inference on CPU and GPU
5. Download the VoxCeleb1 verification dataset
6. Run the full VoxCeleb1 EER benchmark on CPU and GPU

Select the **ov_pyan** kernel for this notebook before running any cell below.

## 1. Create the Conda environment

This step cannot be a notebook cell: the kernel this notebook runs on **is** `ov_pyan`, so it cannot create or recreate itself. Run the following once, in a terminal, **before** selecting the `ov_pyan` kernel for this notebook. It checks whether `ov_pyan` already exists and only creates it if missing:

```bash
cd /home/user/ov_pyannote-embedding
if conda env list | grep -qE '^\s*pyan_ov\s'; then
  echo "pyan_ov already exists, skipping creation."
else
  conda env create -f pyan_ov.yaml
fi
```

To update an existing environment instead:

```bash
conda env update -n pyan_ov -f pyan_ov.yaml --prune
```

Then, in VS Code, click **Select Kernel** (top-right of this notebook) and choose **ov_pyan**.

In [ ]:
from pathlib import Path
import os

working_dir = Path.cwd()
if (working_dir / "convert_to_openvino.py").is_file() and (working_dir / "notebook").is_dir():
    working_dir = working_dir / "notebook"
os.chdir(working_dir)

import openvino as ov

print("Working directory:", Path.cwd())
print("OpenVINO version:", ov.__version__)
print("Available devices:", ov.Core().available_devices)

## 2. Hugging Face access (one time)

**Step 1 — Create an access token:**

1. Go to https://huggingface.co/settings/tokens
2. Click **New token** → choose type **Read** → give it a name → **Create**.
3. Copy the token (starts with `hf_...`).

**Step 2 — Accept model terms** (once per model):

- https://huggingface.co/pyannote/embedding

**Step 3 — Login with your token, in a terminal** (not a notebook cell — the prompt needs interactive input):

```bash
huggingface-cli login
# paste your hf_... token when prompted
```

In [ ]:
!huggingface-cli whoami

## 3. Convert the model to OpenVINO IR

`../convert_to_openvino.py` downloads `pyannote/embedding`, traces it with `torch.jit.trace`, and converts it to OpenVINO IR with a dynamic time axis. This notebook saves the generated files inside the notebook folder as `models/pyannote_embedding.xml` and `models/pyannote_embedding.bin`.

In [ ]:
!python convert_to_openvino.py --output-dir models

### Conversion status

Warnings may appear after this cell finishes, including Lightning checkpoint warnings or OpenTelemetry shutdown messages. These warnings do **not** indicate an OpenVINO conversion failure when the output below is present:

```text
Saved IR to: models/pyannote_embedding.xml
          and models/pyannote_embedding.bin
```

The next cell checks that both generated files exist. If both files are found, the model conversion completed successfully.

In [ ]:
from pathlib import Path

xml_path = Path("models/pyannote_embedding.xml")
bin_path = Path("models/pyannote_embedding.bin")
assert xml_path.exists() and bin_path.exists(), "Run the conversion cell above first."
print(f"{xml_path} ({xml_path.stat().st_size / 1024:.1f} KB)")
print(f"{bin_path} ({bin_path.stat().st_size / 1024 / 1024:.1f} MB)")

## 4. Run sample embedding inference

`infer_openvino.py` loads the IR, embeds each WAV file to a 512-d speaker vector, and prints the cosine distance between the two files (smaller = more likely the same speaker).

### CPU

In [ ]:
!python infer_openvino.py samples/speakerA.wav samples/speakerB.wav --model models/pyannote_embedding.xml --device CPU

### GPU

Use `GPU` for a single GPU device, or `GPU.0` / `GPU.1` when OpenVINO reports more than one (see the device list printed in Cell 3). Explicit `--precision f16` uses one dynamic-shape compiled model; explicit `--precision f32` requires a static shape and is compiled per audio length.

In [ ]:
!python infer_openvino.py samples/speakerA.wav samples/speakerB.wav --model models/pyannote_embedding.xml --device GPU --precision f16

## 5. Download the VoxCeleb1 verification dataset

`../download_voxceleb_test.sh` downloads the trial pair list (`veri_test.txt`, ~1.5 MB) and the VoxCeleb1 test WAVs (~1 GB) into `test_audio/` inside this notebook folder. It is idempotent -- it skips anything already downloaded. Run the Hugging Face login cell above before this if you have not already.

In [ ]:
!../download_voxceleb_test.sh test_audio

In [ ]:
TRIALS = Path("test_audio/veri_test.txt")
WAV_ROOT = Path("test_audio/vox1/wav")
assert TRIALS.is_file() and WAV_ROOT.is_dir(), "Run the download cell above first."
print("Trial list:", TRIALS.resolve())
print("WAV root:  ", WAV_ROOT.resolve())

## 6. Run the full VoxCeleb1 EER benchmark

`benchmark_eer.py` embeds all 4,715 unique clips referenced by the 37,720 trial pairs, then reports the Equal Error Rate. The model card claims 2.8% EER.

### CPU (FP32)

This can take a few minutes.

In [ ]:
!python benchmark_eer.py --backend openvino --device CPU --precision f32 \
    --trials test_audio/veri_test.txt --wav-root test_audio/vox1/wav

### GPU (FP16)

FP16 on GPU uses one dynamic-shape compiled model, so it stays fast across the many different clip lengths in VoxCeleb1.

In [ ]:
!python benchmark_eer.py --backend openvino --device GPU --precision f16 \
    --trials test_audio/veri_test.txt --wav-root test_audio/vox1/wav